In [ ]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
cols = ["SD_front", "SD_left", "SD_right", "SD_back", "Class"]
df = pd.read_csv("../wall+following+robot+navigation+data/sensor_readings_4_processed.csv")

X = df[cols[:-1]].to_numpy()
classes, y = np.unique(df["Class"], return_inverse=True)

ORDENS = {"1/2": 1/2, "2/3": 2/3, "1": 1, "3/2": 3/2, "2": 2, "5/2": 5/2}
SPLITS = {"20/80": 0.2, "30/70": 0.3, "50/50": 0.5, "70/30": 0.7, "80/20": 0.8}
RODADAS = 100
STATS = ["mean", "std", "max", "min", "median"]

In [ ]:
def treinar(X, y):
    return X, y  # 1-NN apenas memoriza o conjunto de treino


def prever(modelo, X, m):
    Xtr, ytr = modelo
    # raiz de ordem 1/m omitida: é monotônica, não altera o argmin
    d = sum(np.abs(X[:, [k]] - Xtr[:, k]) ** m for k in range(X.shape[1]))
    return ytr[d.argmin(axis=1)]


def separar(y, frac, rng):
    # divisão estratificada: mesma proporção de cada classe no treino e no teste
    tr = np.concatenate([rng.permutation(np.flatnonzero(y == c))[:round(frac * np.sum(y == c))]
                         for c in np.unique(y)])
    return tr, np.setdiff1d(np.arange(len(y)), tr)


def matriz_confusao(y_true, y_pred, k):
    return np.bincount(y_true * k + y_pred, minlength=k * k).reshape(k, k)

In [ ]:
rng = np.random.default_rng(42)
registros, confusoes = [], {}

for split, frac in SPLITS.items():
    for r in range(RODADAS):
        tr, te = separar(y, frac, rng)
        for nome, m in ORDENS.items():
            t0 = time.perf_counter()
            modelo = treinar(X[tr], y[tr])
            t1 = time.perf_counter()
            pred = prever(modelo, X[te], m)
            t2 = time.perf_counter()

            cm = matriz_confusao(y[te], pred, len(classes))
            confusoes[split, nome, r] = cm
            registros.append({
                "split": split, "m": nome, "rodada": r,
                "acuracia": cm.trace() / cm.sum(),
                "t_treino": t1 - t0, "t_teste": t2 - t1,
                **dict(zip(classes, cm.diagonal() / cm.sum(axis=1))),
            })

res = pd.DataFrame(registros)

In [ ]:
grupos = res.groupby(["split", "m"], sort=False)
tabela = grupos["acuracia"].agg(STATS).join(grupos[["t_treino", "t_teste"]].mean())
display(tabela.round(4))

In [ ]:
plt.figure(figsize=(10, 5))
sns.pointplot(res, x="split", y="acuracia", hue="m", errorbar="sd", dodge=0.5)
plt.title("Acurácia (média ± desvio-padrão) por separação treino/teste");

In [ ]:
SPLIT_REF = "80/20"
ref = res[res["split"] == SPLIT_REF]

fig, axes = plt.subplots(len(ORDENS), 2, figsize=(11, 4.5 * len(ORDENS)))
for linha, nome in zip(axes, ORDENS):
    sub = ref[ref["m"] == nome]
    for ax, (rotulo, i) in zip(linha, [("Melhor", sub["acuracia"].idxmax()), ("Pior", sub["acuracia"].idxmin())]):
        cm = confusoes[SPLIT_REF, nome, sub.loc[i, "rodada"]]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=classes, yticklabels=classes, ax=ax)
        ax.set(title=f"m = {nome} | {rotulo} rodada (acc = {sub.loc[i, 'acuracia']:.4f})",
               xlabel="Predito", ylabel="Real")
plt.tight_layout()

In [ ]:
por_classe = res.melt(id_vars=["split", "m"], value_vars=list(classes),
                      var_name="classe", value_name="acuracia_classe")

for classe, g in por_classe.groupby("classe"):
    print(classe)
    display(g.groupby(["split", "m"], sort=False)["acuracia_classe"].agg(STATS).round(4))

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(por_classe[por_classe["split"] == SPLIT_REF], x="classe", y="acuracia_classe", hue="m")
plt.title(f"Acurácia por classe ({SPLIT_REF})");